In [3]:
! pip install ollama

In [ ]:
import os
import sys
import json
import pandas as pd
from config import (
    PATH_CLEAN_RECIPES,
    PATH_CLEAN_INTERACTIONS
)

# CORREZIONE PERCORSI: Diciamo a Python di partire dalla cartella superiore '..'
# In questo modo 'dataset/clean_recipes.csv' diventa '../dataset/clean_recipes.csv'
PATH_RECIPES_CORRECT = os.path.join('..', PATH_CLEAN_RECIPES)
PATH_INTERACTIONS_CORRECT = os.path.join('..', PATH_CLEAN_INTERACTIONS)

# Ora carichiamo i dati con i percorsi corretti rispetto alla posizione del notebook
df_recipes = pd.read_csv(PATH_RECIPES_CORRECT)
df_interactions = pd.read_csv(PATH_INTERACTIONS_CORRECT)

# 1. FORZIAMO IL PERCORSO RADICE (Root Directory)
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), ".."))) 

# 2. IMPORT DI TUTTI I MODELLI, METRICHE E LAYER LLM
from models.popularity import PopularityRecommender
from models.content_based import ContentBasedRecommender
from models.health_based import HealthBasedRecommender
from models.mood_based import MoodBasedRecommender
from models.collaborative_filtering import CollaborativeFilteringRecommender
from models.hybrid_based import HybridRecommender  # File: hybrid_based.py
from llm.intent_parser import LLMIntentParser
from llm.explainer import LLMExplainer

# 3. INIZIALIZZAZIONE E FIT ATOMICO DEI 5 PILASTRI
print("-> Addestramento dei 5 modelli di base...")

pop_model = PopularityRecommender(m=50)
pop_model.fit(df_recipes, df_interactions)

content_model = ContentBasedRecommender(matrix_path="../dataset/tfidf_matrix.npz")
content_model.fit(df_recipes)

health_model = HealthBasedRecommender()
health_model.fit(df_recipes)

mood_model = MoodBasedRecommender()
mood_model.fit(df_recipes)

cf_model = CollaborativeFilteringRecommender(min_user_interactions=5)
cf_model.fit(df_recipes, df_interactions)

# 4. INIZIALIZZAZIONE DELL'IBRIDO PASSANDO I 5 MODELLI RICHIESTI (Risolve il TypeError!)
hybrid_model = HybridRecommender(pop_model, content_model, mood_model, cf_model, health_model)

# 5. INIZIALIZZAZIONE DEI MODULI CON LLAMA
parser = LLMIntentParser(model_name="llama3")
explainer = LLMExplainer(model_name="llama3")

print("\n" + "="*60 + "\n")
print("--- TEST COMPLETO: INTENT PARSING -> HYBRID REC -> EXPLAINER ---")

# 6. RICHIESTA IN INGRESSO
USER_QUERY = "Sono esausto dopo lo studio, voglio qualcosa di super veloce e confortevole. Ho del chicken in frigo e vorrei stare sotto le 500 calorie!"
print(f"🗣️ UTENTE: '{USER_QUERY}'\n")

# 7. FASE 1: PARSING LOCALE CON LLAMA
parsed_json = parser.parse_query(USER_QUERY)
print("🤖 JSON ESTRATTO DA LLM (LLAMA 3):")
print(json.dumps(parsed_json, indent=2))
print("\n" + "-"*50 + "\n")

# 8. FASE 2: RACCOMANDAZIONE IBRIDA 
ricette_consigliate = hybrid_model.recommend(
    user_ingredients=parsed_json.get('ingredients'),
    mood_params=parsed_json.get('mood'),
    health_params={
        'profile_name': 'weight_loss',
        'max_calories': parsed_json.get('max_calories')
    },
    top_k=1
)

# 9. FASE 3: SPIEGAZIONE LOCALE CON LLAMA
if ricette_consigliate:
    top_ricetta = ricette_consigliate[0]
    print(f"🏆 ALGORITMO IBRIDO HA SCELTO: {top_ricetta['name'].upper()}")
    print(f"   (Dettagli: {top_ricetta['calorie']} kcal, {top_ricetta['minuti']} min)")
    print("\n" + "-"*50 + "\n")
    
    spiegazione_piatto = explainer.generate_explanation(
        original_query=USER_QUERY,
        recipe_name=top_ricetta['name'],
        recipe_details=top_ricetta
    )
    print(f"👨‍🍳 CHEF AI (LLAMA 3) SPIEGA:\n{spiegazione_piatto}")
else:
    print("Nessuna ricetta trovata per i criteri indicati.")

TypeError: HybridRecommender.__init__() missing 5 required positional arguments: 'popularity_model', 'content_model', 'mood_model', 'cf_model', and 'health_model'